# Advisory text preprocessing and NMF topic model
Input: `../Step1_crawler/unstructureddataset.xlsx` (123 raw CISA advisories, scraped earlier; nothing is crawled here).
Outputs: `structureddataset.xlsx` (128 advisories) and the NMF topics (9 topics, `random_state=42`).

## 1. Load raw advisories

In [13]:
import pandas as pd
import numpy as np
import re

df = pd.read_excel('../Step1_crawler/unstructureddataset.xlsx', engine='openpyxl')
print('raw advisories:', df.shape[0])
structureddataset = df

raw advisories: 123


## 2. Extract CVE / CWE IDs, remove the row without a CVE

In [14]:
# Pull every CVE ID out of a piece of text
def extract_cve_ids(text):
    return re.findall(r'CVE-\d{4}-\d{4,7}', text)

# Pull every CWE ID out of a piece of text
def extract_cwe_ids(text):
    # Match CWE IDs even if there's no space between "CWE-" and the number
    return re.findall(r'CWE-\d+', text)

# Add the CVE and CWE columns from the vulnerability overview
structureddataset['CVE_IDs'] = structureddataset['Vulnerability Overview'].apply(lambda x: ', '.join(extract_cve_ids(x)))
structureddataset['CWE_IDs'] = structureddataset['Vulnerability Overview'].apply(lambda x: ', '.join(extract_cwe_ids(x)))

def remove_duplicates(cell):
    if pd.isna(cell):
        return cell
    # Lowercase the IDs and drop repeats within a cell
    ids = cell.split(',')
    unique_ids = list(set(id_.strip().lower() for id_ in ids))
    return ','.join(unique_ids)

# Do this for both ID columns
structureddataset['CVE_IDs'] = structureddataset['CVE_IDs'].apply(remove_duplicates)
structureddataset['CWE_IDs'] = structureddataset['CWE_IDs'].apply(remove_duplicates)

structureddataset = structureddataset[[ "URL", "CVSS", "Vendor", "Equipment", "Vulnerability", "Risk Evaluated", "Affected Product", "Vulnerability Overview", "Researcher", "Mitigation", "CVE_IDs", "CWE_IDs"]]
structureddataset = structureddataset.drop(index=121)  # this advisory has no CVE ID
structureddataset = structureddataset.reset_index(drop=True)
structureddataset

,URL,CVSS,Vendor,Equipment,Vulnerability,Risk Evaluated,Affected Product,Vulnerability Overview,Researcher,Mitigation,CVE_IDs,CWE_IDs
0,https://www.cisa.gov/news-events/ics-medical-a...,Critical,Philips,Vue PACS,"Out-of-bounds Write, Deserialization of Untrus...",Successful exploitation of these vulnerabiliti...,The following Philips products are affected: V...,3.2.1OUT-OF-BOUNDS WRITE CWE-787 A third-party...,TAS Health NZ and Camiel van Es reported these...,Philips recommends the following mitigations: ...,"cve-2023-40704,cve-2023-40159,cve-2020-10673,c...","cwe-200,cwe-400,cwe-1392,cwe-502,cwe-521,cwe-7..."
1,https://www.cisa.gov/news-events/ics-medical-a...,High,MicroDicom,DICOM Viewer,Improper Authorization in Handler for Custom U...,Successful exploitation of these vulnerabiliti...,The following versions of MicroDicom DICOM Vie...,3.2.1IMPROPER AUTHORIZATION IN HANDLER FOR CUS...,Michael Heinzl reported these vulnerabilities ...,MicroDicom recommends users upgrade to DICOM V...,"cve-2024-33606,cve-2024-28877","cwe-939,cwe-121"
2,https://www.cisa.gov/news-events/ics-medical-a...,Critical,Baxter,Welch Allyn Configuration Tool,Insufficiently Protected Credentials,Successful exploitation of this vulnerability ...,The following Baxter (formerly Hillrom and Wel...,3.2.1INSUFFICIENTLY PROTECTED CREDENTIALS CWE-...,Baxter reported this vulnerability to CISA.,Baxter has found no evidence to date of any co...,cve-2024-5176,cwe-522
3,https://www.cisa.gov/news-events/ics-medical-a...,Critical,Baxter,Welch Allyn Connex Spot Monitor (CSM),Use of Default Cryptographic Key,Successful exploitation of this vulnerability ...,The following Baxter (formerly manufactured by...,3.2.1USE OF DEFAULT CRYPTOGRAPHIC KEY CWE-1394...,Maarten Boone and Edwin Van Andel (CTO of Zero...,Baxter has released a software update for all ...,cve-2024-1275,cwe-1394
4,https://www.cisa.gov/news-events/ics-medical-a...,High,Santesoft,Sante FFT Imaging,Out-of-Bounds Write,Successful exploitation of this vulnerability ...,The following Santesoft products are affected:...,3.2.1OUT-OF-BOUNDS WRITE CWE-787 On Santesoft ...,Michael Heinzl reported this vulnerability to ...,Santesoft released an updated version of their...,cve-2024-1696,cwe-787
...,...,...,...,...,...,...,...,...,...,...,...,...
117,https://www.cisa.gov/news-events/ics-medical-a...,Medium,Medtronic,N’Vision Clinician Programmer,"Missing Encryption of Sensitive Data, Protecti...",As part of the normal functionality of this de...,--------- Begin Update A Part 4 of 5 -------- ...,4.2.1MISSING ENCRYPTION OF SENSITIVE DATA CWE-...,Billy Rios and Jonathan Butts of Whitescope LL...,Medtronic has not developed a product update t...,"cve-2018-8849,cve-2018-10631","cwe-311,cwe-693"
118,https://www.cisa.gov/news-events/ics-medical-a...,High,"Vendors: Silex Technology, GE Healthcare","SX-500, SD-320AN, MobileLink","Improper Authentication, OS Command Injection",Successful exploitation of these vulnerabiliti...,The following products from Silex Technology a...,4.2.1IMPROPER AUTHENTICATION CWE-287 Authentic...,Eric Evenchick of Atredis Partners reported th...,Silex Technologies and GE Healthcare recommend...,"cve-2018-6021,cve-2018-6020","cwe-287,cwe-78"
119,https://www.cisa.gov/news-events/ics-medical-a...,Medium,"Becton, Dickinson and Company (BD)",Certain BD Pyxis Products,Reusing a Nonce,Successful exploitation of this vulnerability ...,"The following versions of BD Pyxis products, a...","3.2.1REUSING A NONCE, KEY PAIR IN ENCRYPTION C...","Mathy Vanhoef of imec-DistriNet, KU Leuven dis...",BD has implemented third-party vendor patches ...,"cve-2017-13078,cve-2017-13079,cve-2017-13080,c...",cwe-323
120,https://www.cisa.gov/news-events/ics-medical-a...,High,Abbott Laboratories,Implantable Cardioverter Defibrillator and Car...,Improper Authentication and Improper Restricti...,Successful exploitation of these vulnerabiliti...,The following ICDs and CRT-Ds manufactured and...,3.2.1IMPROPER AUTHENTICATION CWE-287 The de

## 3. Split multi-vendor rows (row 81: ENEA / Green Hills / ITRON / IP Infusion; row 122: Silex / GE Healthcare)

In [15]:
import pandas as pd

# Row 81 lists several vendors and devices in one advisory
row_81 = structureddataset.loc[81]

# Separate the vendor and equipment lists
vendors = row_81['Vendor'].replace('Vendors:', '').split(', ')
equipment = row_81['Equipment'].replace('and ', '').split(', ')

# One row per vendor and device, other columns copied from the original row
structure_divided = pd.DataFrame({
    'URL': row_81['URL'],
    'CVSS': row_81['CVSS'],
    'Vendor': vendors,
    'Equipment': equipment,
    'Vulnerability': row_81['Vulnerability'],
    'Risk Evaluated': row_81['Risk Evaluated'],
    'Affected Product': row_81['Affected Product'],
    'Vulnerability Overview': row_81['Vulnerability Overview'],
    'Researcher': row_81['Researcher'],
    'Mitigation': row_81['Mitigation'],
    'CVE_IDs': row_81['CVE_IDs'],
    'CWE_IDs': row_81['CWE_IDs']
})

structure_divided

# Replace the original row with the split rows
structureddataset = structureddataset.drop(81).reset_index(drop=True)

# Put the split rows back at the same position
structureddataset = pd.concat([structureddataset.iloc[:81], structure_divided, structureddataset.iloc[81:]]).reset_index(drop=True)

In [16]:
# Row 122 is one advisory for two vendors and three devices
vendors = ['Silex Technology', 'Silex Technology', 'GE Healthcare']
equipment = ['SX-500', 'SD-320AN', 'MobileLink']

# Build one row per device
new_rows = []

# Pair each device with its vendor
for v, e in zip(vendors, equipment):
    new_row = structureddataset.loc[122].copy()  # copy of the original row
    new_row['Vendor'] = v  # vendor for this device
    new_row['Equipment'] = e  # the device
    new_rows.append(new_row)

# Collect the new rows in a DataFrame
structure_divided1 = pd.DataFrame(new_rows)

# Replace the original row with the split rows
structureddataset = structureddataset.drop(122).reset_index(drop=True)

# Put the split rows back at the same position
structureddataset = pd.concat([structureddataset.iloc[:122], structure_divided1, structureddataset.iloc[122:]]).reset_index(drop=True)

## 4. Standardise vendor names

In [17]:
# One standard name per vendor
vendor_mapping = {
    'Becton': 'Becton, Dickinson and Company (BD)',
    'Dickinson and Company (BD)': 'Becton, Dickinson and Company (BD)',
    '(BD) Becton': 'Becton, Dickinson and Company (BD)',
    'GE': 'GE Healthcare',
    'GE Healthcare': 'GE Healthcare',
    'Inc. (Natus)': 'Natus Medical',
    'Hillrom Medical. Welch Allyn, and ELI are registered trademarks of Baxter International, Inc., or its subsidiaries': 'Baxter',
    'Hillrom Medical. Welch Allyn': 'Baxter',
    'Hillrom Medical. Welch Allyn and ELI are registered trademarks of Baxter International': 'Baxter',
    'Hillrom': 'Baxter',
    'Natus Medical, Inc. (Natus)': 'Natus Medical',
    'OpenClinic GA is a product of open-source collaboration on Source Forge': 'Source Forge',
    'Hillrom Medical. Welch Allyn, and ELI are registered trademarks of Baxter International, Inc.': 'Baxter',
    'Hillrom Medical. Welch Allyn, and ELI are registered trademarks of Baxter International, Inc., or its subsidiaries.': 'Baxter',
    'Natus Medical': 'Natus Medical',
    '(BD) Becton, Dickinson and Company': 'Becton, Dickinson and Company (BD)'
}

# Apply the mapping
structureddataset['Vendor'] = structureddataset['Vendor'].replace(vendor_mapping)

# Advisories per vendor after standardising
vendor_counts = structureddataset['Vendor'].value_counts()

vendor_counts

Vendor
Philips                               32
Becton, Dickinson and Company (BD)    19
Medtronic                             14
Baxter                                10
GE Healthcare                          5
B. Braun Melsungen AG                  4
Santesoft                              4
MicroDicom                             2
Change Healthcare                      2
Silex Technology                       2
​Softneta                              1
Orthanc                                1
Contec Health                          1
AliveCor                               1
Illumina                               1
OFFIS                                  1
Swisslog Healthcare                    1
Ypsomed                                1
Fresenius Kabi                         1
LifePoint Informatics                  1
ZOLL                                   1
Hamilton Medical AG                    1
SOOIL Developments Co., Ltd.           1
Innokas Yhtymä Oy                      1
Capsule T

## 5. Clean the text columns

In [18]:
import re
# Remove the vulnerability overview text that got mixed into Affected Product
def remove_matching_textAV(row):
    affected_product = row['Affected Product']
    vulnerability_overview = row['Vulnerability Overview']

    risk_evaluated  = row['Risk Evaluated']
 
    # Pattern that matches the overview text exactly
    pattern = re.escape(vulnerability_overview)
    cleaned_text = re.sub(pattern, '', affected_product, flags=re.DOTALL).strip()
     # Also remove headings such as "4.2 VULNERABILITY OVERVIEW"
    pattern_variation = r'\d+\.\d+ VULNERABILITY OVERVIEW'
    cleaned_text = re.sub(pattern_variation, '', cleaned_text, flags=re.IGNORECASE | re.DOTALL).strip()
      # Remove text between rows of dashes
    pattern_dashed_lines = r'-+\s*.*?\s*-+'
    cleaned_text = re.sub(pattern_dashed_lines, '', cleaned_text, flags=re.DOTALL)
    return cleaned_text

# Apply to every row
structureddataset['Affected Product'] = structureddataset.apply(remove_matching_textAV, axis=1)

In [19]:
def remove_matching_textRE(row):
    risk_evaluated  = row['Risk Evaluated']

 
    pattern_dashed_lines = r'-+\s*.*?\s*-+'
    cleaned_text = re.sub(pattern_dashed_lines, '', risk_evaluated, flags=re.DOTALL)
    return cleaned_text

# Apply to every row
structureddataset['Risk Evaluated'] = structureddataset.apply(remove_matching_textRE, axis=1)

In [20]:
import re

def clean_and_process_text(row):
    vulnerability_overview = row['Vulnerability Overview']

    # Replace newlines and tabs with spaces
    vulnerability_overview = re.sub(r'[\n\t\r]', ' ', vulnerability_overview)

    # Cut everything after the word BACKGROUND
    vulnerability_overview = re.sub(r'\bBACKGROUND\b.*', '', vulnerability_overview)

    # Remove section numbers such as 3.2.1
    vulnerability_overview = re.sub(r'\b\d+\.\d+(?:\.\d+)?(?=\s|[A-Z])', '', vulnerability_overview)

    # Split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', vulnerability_overview)

    # Drop sentences that mention CVSS
    sentences = [sentence for sentence in sentences if 'CVSS' not in sentence]

    # Join the remaining sentences
    cleaned_text = ' '.join(sentences)

    cleaned_text = re.sub(r'(\d)(h)', r'\1 \2', cleaned_text)
    # Tidy spaces and repeated punctuation
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    cleaned_text = re.sub(r' +', ' ', cleaned_text)  # collapse repeated spaces
    cleaned_text = re.sub(r'([,;:\.]){2,}', r'\1', cleaned_text)  # collapse repeated punctuation
    cleaned_text = re.sub(r'-{2,}\s*.*?\s*-{2,}', '', cleaned_text, flags=re.DOTALL)

    return cleaned_text

# Apply to every row
structureddataset['Vulnerability Overview'] = structureddataset.apply(clean_and_process_text, axis=1)

In [21]:
print('advisories after cleaning:', structureddataset.shape[0])
structureddataset.to_excel('structureddataset.xlsx')

advisories after cleaning: 128


## 6. TF-IDF and NMF topic model

In [22]:
df = structureddataset

In [23]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
# Join the text columns of each advisory into one string
df['Combined_Text'] = df[['URL', 'CVSS', 'Vendor', 'Equipment', 'Vulnerability',
                          'Risk Evaluated', 'Affected Product', 'Vulnerability Overview',
                          'Researcher', 'Mitigation']].apply(lambda x: ' '.join(x.astype(str)), axis=1)

def clean_text(text):
    # Keep letters only
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    # Lowercase
    text = text.lower()
    # Collapse spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Drop words shorter than 3 letters
    words = [word for word in text.split() if len(word) >= 3]
    return ' '.join(words)

# Clean the combined text
df['Cleaned_Combined_Text'] = df['Combined_Text'].apply(clean_text)

# TF-IDF on word n-grams of length 1 to 6, top 1000 terms
vectorizer = TfidfVectorizer(stop_words='english', ngram_range=(1, 6), max_features=1000,norm='l2')
X = vectorizer.fit_transform(df['Cleaned_Combined_Text'])

# NMF topic model
num_topics = 9
nmf_model = NMF(n_components=num_topics, random_state=42, beta_loss='frobenius')
W = nmf_model.fit_transform(X)  # document by topic
H = nmf_model.components_  # topic by term

# Print the top words of each topic
def display_topics(H, feature_names, num_top_words):
    for topic_idx, topic in enumerate(H):
        print(f"Topic {topic_idx + 1}:")
        top_indices = topic.argsort()[:-num_top_words - 1:-1]
        top_words = [feature_names[i] for i in top_indices]
        print(", ".join(top_words))

num_top_words = 8
feature_names = vectorizer.get_feature_names_out()
display_topics(H, feature_names, num_top_words)

Topic 1:
cisa, social, social engineering, engineering, email, vulnerability, ics, avoiding
Topic 2:
nccic, remote, vpns, networks, vulnerabilities, secure, devices, version
Topic 3:
philips, vulnerability, service, service support, security, users, philips recommends, information
Topic 4:
buffer, vxworks, tcp, stack, buffer overflow, overflow, argument, cve
Topic 5:
medtronic, programmer, nccic, patient, vulnerabilities, monitor, device, affected
Topic 6:
alaris, unit, wireless, vulnerability, network, authentication, nccic, systems
Topic 7:
vulnerabilities, access, credentials, cisa, device, hard, hard coded, coded
Topic 8:
pyxis, key, vulnerability, credentials, management, access, products, systems
Topic 9:
insulin, pump, wireless, attackers, vulnerability, settings, medtronic, communication


In [24]:
df['Dominant_Topic'] = W.argmax(axis=1) + 1  
df['Dominant_Topic_Weight'] = W.max(axis=1)

topic_counts = df['Dominant_Topic'].value_counts().sort_index()
topic_prevalence = topic_counts / len(df) * 100  # share of advisories per topic (%)
print(topic_prevalence)

Dominant_Topic
1     5.46875
2    20.31250
3    21.09375
4     7.03125
5     7.81250
6     4.68750
7    25.78125
8     3.90625
9     3.90625
Name: count, dtype: float64
